In [4]:
# %%
import openai
from tqdm.auto import tqdm
import os
import time
openai.api_key = os.environ["OPENAI_API_KEY"]

# %%
import sys
sys.path.append('../../..')
print(os.path.realpath("."))

from data.dataset import ReimburseGraphDataset, StandardGraphDataset, DataAugmentationLevel, NodeType, DialogNode, Question

# %%
onboard_human_data = StandardGraphDataset('en/onboarding/train_graph.json', 'en/onboarding/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')

# %%
def parse_output(result):
    result_strings = result.get('choices')[0].get("message").get("content").split('\n')
    questions = []
    unnumbered_questions = []
    question_idx = 1
    for question in result_strings:
        question = question.replace('"\n', "").replace('\"', "").strip()
        if question.startswith(f"{question_idx}."):
            questions.append(question.strip(f"{question_idx}.").strip())
        else:
            unnumbered_questions.append(question)
        question_idx += 1
    return questions, unnumbered_questions


/mount/arbeitsdaten/asr-2/vaethdk/virtualenvs/cts_en/lib64/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/mount/arbeitsdaten41/projekte/asr-2/vaethdk/cts_newcodebase_rollback/conversational-tree-search/generation/onboarding/chatgpt
===== Dataset Statistics =====
- files:  en/onboarding/train_graph.json en/onboarding/train_answers.json
- synonyms: True
- depth: 12  - degree: 9
- answers: 70
- questions: 141
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  4
- answer limit: 0  - maximum loaded:  1


In [7]:

def prompt(node_text: str, answer_text: str, num_paraphrases: int):
    return f"""Generate {num_paraphrases} paraphrases for the response "{answer_text}" to the question {node_text}"""

def api_prompt(prompt: str):
    return [
        {"role": "system", "content": "You are generating semantically similar paraphrases for a given response to some question. The generated response paraphrases should be human-like and short, using frequently used words and phrases only. Present the results in a numbered list."},
        {"role": "user", "content": prompt},
    ]

def api_completion(node_text: str, answer_text: str, num_paraphrases: int):
    return openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=api_prompt(prompt(node_text, answer_text, num_paraphrases))
    )


from collections import defaultdict
import traceback
NUM_PARAPHRASES = 5

generated = defaultdict(lambda: set())
generated_unnumbered = defaultdict(lambda: set())

num_generated = 0
num_generated_unnumbered = 0

for idx, node in tqdm(enumerate(onboard_human_data.nodes_by_type[NodeType.QUESTION])):
    for answer in node.answers:
        done = False
        while not done:
            try:
                response = api_completion(node.text, answer.text, NUM_PARAPHRASES)
                answers, unnumbered_answers = parse_output(response)

                generated[answer.key] = generated[answer.key].union(answers)
                generated_unnumbered[answer.key] = generated_unnumbered[answer.key].union(unnumbered_answers)

                num_generated += len(answers)
                num_generated_unnumbered += len(unnumbered_answers)

                if idx % 10 == 0:
                    print(f"Generated: {num_generated}, Unnumbered: {num_generated_unnumbered}")
                
                done = True
            except:
                # traceback.print_exc()
                print("waiting...")
                time.sleep(15)
        break
    break

print(generated)

0it [00:04, ?it/s]

Generated: 5, Unnumbered: 0
defaultdict(<function <lambda> at 0x7f496e97a3b0>, {16939912049114625: {'Exploring housing opportunities', 'Looking for housing options', 'House hunting', 'Trying to secure a place to live', 'Searching for accommodation'}})


In [8]:
   
# %% Keyword-based generation

def prompt(node_text: str, answer_text: str, num_paraphrases: int):
    return f"""Generate {num_paraphrases} options for shortening the response "{answer_text}" to the question {node_text}"""

def api_prompt_keywords(prompt: str):
    return [
        {"role": "system", "content": "You are shortening a given response to some question into a keyword-like prompt. Present the results in a numbered list."},
        {"role": "user", "content": prompt},
    ]

def api_completion_keywords(node_text: str, answer_text: str, num_paraphrases: int):
    return openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=api_prompt_keywords(prompt(node_text, answer_text, num_paraphrases))
    )

NUM_KEYWORD_PARAPHRASES = 5

print("1", generated)

for idx, node in tqdm(enumerate(onboard_human_data.nodes_by_type[NodeType.QUESTION])):
    for answer in node.answers:
        done = False
        while not done:
            try:
                response = api_completion_keywords(node.text, answer.text, NUM_KEYWORD_PARAPHRASES)
                answers, unnumbered_answers = parse_output(response)

                generated[answer.key] = generated[answer.key].union(answers)
                print("2", answers)
                generated_unnumbered[answer.key] = generated_unnumbered[answer.key].union(unnumbered_answers)

                num_generated += len(answers)
                num_generated_unnumbered += len(unnumbered_answers)

                if idx % 10 == 0:
                    print(f"Generated: {num_generated}, Unnumbered: {num_generated_unnumbered}")
                
                done = True
            except:
                traceback.print_exc()
                done = True
                print("waiting...")
                time.sleep(15)
        break
    break

print("3", generated)

1 defaultdict(<function <lambda> at 0x7f496e97a3b0>, {16939912049114625: {'Exploring housing opportunities', 'Looking for housing options', 'House hunting', 'Trying to secure a place to live', 'Searching for accommodation'}})


0it [00:04, ?it/s]

2 ['Housing search', 'Housing assistance', 'Stuttgart housing', 'Finding a home', 'Accommodation in Stuttgart']
Generated: 10, Unnumbered: 0
3 defaultdict(<function <lambda> at 0x7f496e97a3b0>, {16939912049114625: {'Accommodation in Stuttgart', 'Finding a home', 'Housing assistance', 'Exploring housing opportunities', 'Housing search', 'Looking for housing options', 'House hunting', 'Searching for accommodation', 'Stuttgart housing', 'Trying to secure a place to live'}})


In [9]:
   

import json
with open("../../../resources/en/onboarding/generated/chatgpt/train_answers_v2.json", "w") as f:
    formatted = {}
    for answer_key in generated:
        formatted[answer_key] = list(generated[answer_key])
    json.dump(formatted, f)

with open("../../../resources/en/onboarding/generated/chatgpt/train_answers_unnumbered_v2.json", "w") as f:
    formatted = {}
    for answer_key in generated_unnumbered:
        formatted[answer_key] = list(generated_unnumbered[answer_key])
    json.dump(formatted, f)
